**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**04 | MATLAB-Native Conversion and Biomechanical Signal Validation**

**Research question:** Can the original Essex MATLAB object-encoded `IMU` and model-derived `e1e3` records be converted to traceable, finite numerical arrays without inventing channel labels, target units or temporal alignment?

**Original source:** Hollinger et al. (2026), *Scientific Data*, [10.1038/s41597-026-07429-3](https://doi.org/10.1038/s41597-026-07429-3). The pinned dataset is [Figshare 30225739 v13](https://doi.org/10.6084/m9.figshare.30225739.v13); 40 participants and 1,897 original `Data` records were independently audited in Notebooks 01–03.

**Two complementary workflows:** (1) an optional MATLAB-native conversion script extracts numeric tables without interpolation and preserves original field names, and (2) this Python/Colab notebook validates the result against the prior source registry. The 444-record BoneNet derivative is **not** substituted for the original six-activity source.

**Critical runtime limitation:** MATLAB-native table conversion requires MATLAB Online/Desktop, which is not part of an ordinary Python Colab environment. When no converted ZIP has been provided, this notebook runs a complete **pre-conversion audit**, records `AWAITING_NATIVE_EXPORT` and does not fabricate numeric signals. When the ZIP is provided, the same notebook verifies every exported numeric MAT payload.

**Scientific boundary:** Numeric extraction does not by itself verify the 42 IMU channel meanings or physical units, native `e1/e3` strain definitions, the FE sampling grid or correct input–target temporal correspondence. No model training, FE interpolation or participant-held-out evaluation occurs.

**Figure policy:** The final scientific SVG is added after checking executed outputs. Its visible heading will not contain a stage or notebook number.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.split("/")[-1].removesuffix(".git")

%cd /content

if not repo_path.exists():
    !git clone -q {repo_url}

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

**01. Imports and Environment**

In [ ]:
# ======================== 01. Imports and Environment ========================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

# ---------------- 2. Standard Library and Scientific Packages ----------------
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from IPython.display import display
import matplotlib.pyplot as plt

# ---------- 3. Reusable Scientific and Native-Conversion Utilities -----------
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab
from essex_native_conversion import (
    EXPORT_COLUMNS, find_native_archive, sha256_file,
    unpack_native_archive, validate_native_manifest,
    validate_numeric_payloads, conversion_summary,
)

plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4))

# ------------------- 4. Source and Study Output Locations --------------------
study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
export_dir = raw_dir / "native_export_v13"
output_dir = study_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print("Study:", study_dir.name)
print("MATLAB conversion source:", study_dir / "code/matlab/export_essex_native_tables.m")
print("Research output directory:", output_dir)

**02. Frozen Provenance and Original Participant Registry**

The original six-activity Essex source is identified by its pinned Figshare article version and the SHA256 previously saved in the Notebook 02 and 03 reports. The approved record registry contains `subject_id`, the original file name, a zero-based record index, activity, original `UpdatedTrialID` and FE representation status.

The source integrity gate and the numeric-conversion gate are **independent**. A correctly aligned register does not establish that MATLAB tables have been numerically recovered.

In [ ]:
# =========== 02. Frozen Figshare and Participant Identity Contract ===========

# -------------------- 1. Read Approved Source Audit Files --------------------
prior02_path = output_dir / "02_record_provenance_and_quality_gate.json"
prior03_path = output_dir / "03_numeric_recovery_quality_gate.json"
registry_path = output_dir / "02_original_record_registry.csv"
for p in [prior02_path, prior03_path, registry_path]:
    if not p.is_file():
        raise FileNotFoundError(f"Required approved source evidence missing: {p}")
prior02 = json.loads(prior02_path.read_text(encoding="utf-8"))
prior03 = json.loads(prior03_path.read_text(encoding="utf-8"))
registry = pd.read_csv(
    registry_path,
    dtype={"subject_id":"string", "source_file":"string",
           "updated_trial_id":"string", "task":"string",
           "local_record_key":"string"},
)

# ---------------------- 2. Enforce Exact Source Lineage ----------------------
assert prior02["source_article_id"] == prior03["figshare_article"] == 30225739
assert prior02["version"] == prior03["dataset_version"] == 13
assert prior02["archive_sha256"] == prior03["archive_sha256"]
assert prior02["source_registry_gate"] == prior03["data_integrity_gate"] == "PASSED"
assert len(registry) == 1897 and registry["subject_id"].nunique() == 40
assert registry["local_record_key"].is_unique
print("Original source record identity: VERIFIED")
print("Original movement records:", len(registry))
print("Original participants:", registry["subject_id"].nunique())

**03. Biomechanical Data and Native Conversion Contract**

The publication describes 7 IMU sensors (3 accelerometer and 3 gyroscope axes per sensor), yielding 42 channels, and 101 positions in the time-normalised movement cycle. FE principal strains `e1/e3` are **model-derived**, calculated more sparsely than the input signals. Importantly, 168 original `e1e3` source fields were empty in the approved structural audit.

The source file `IMU` and `e1e3` objects may be native MATLAB tables. The companion converter preserves table column names and saves finite arrays in ordinary MAT-v7 files. It reports unsupported classes and empty targets rather than silently converting them. The conversion contract is separate from a completed supervised dataset.

In [ ]:
# ============ 03. Published Modality Contract and Missing Targets ============

# ------------- 1. Summarise Original FE Availability by Activity -------------
original_fe = (
    registry.groupby(["task", "fe_representation"])
    .size().rename("records").reset_index()
)
assert original_fe["records"].sum() == 1897
assert registry["fe_representation"].eq("empty").sum() == 168
display(original_fe)

# ------------ 2. Record Required Numerical and Scientific Checks -------------
contract = pd.DataFrame([
    {"modality":"IMU", "records":1897, "published_channels":42,
     "published_time_grid":"101 normalised points",
     "unit_and_axis_verification":"pending"},
    {"modality":"FE", "records":1897, "published_channels":2,
     "published_time_grid":"every 10th position; verify original grid",
     "unit_and_axis_verification":"pending"},
])
contract.to_csv(output_dir / "04_expected_conversion_contract.csv",index=False)
display(contract)

**04. MATLAB-Native Extraction and Colab Handoff**

The study includes the real MATLAB conversion source at `biomechanical_load_estimation/code/matlab/export_essex_native_tables.m`. It must be executed in **MATLAB Online or MATLAB Desktop**, not pasted into a Python code cell. Its native conversion is deliberately strict: only genuine numeric columns are exported, with original names and the original participant/activity/trial identity retained.

MathWorks [MATLAB Online](https://matlab.mathworks.com/) can operate in a browser, including a basic free access tier subject to a monthly time allowance. <Link url="https://www.mathworks.com/products/matlab-online/matlab-online-versions.html">Access details</Link>. The original `MATLAB.zip` file is available from Figshare v13 with published file ID `63190690`.

**Commands to run in MATLAB Online/Desktop, in a clean working folder:**

```matlab
websave('MATLAB.zip','https://ndownloader.figshare.com/files/63190690');
unzip('MATLAB.zip','essex_source');

websave('export_essex_native_tables.m', ...
 'https://raw.githubusercontent.com/RezaSaadatyar/human-movement-intelligence/main/biomechanical_load_estimation/code/matlab/export_essex_native_tables.m');

export_essex_native_tables('essex_source','essex_native_export_v13');
```

The script should produce `native_essex_export.zip` and an original-ID-based `export_manifest.csv`. Upload the ZIP to the **Colab Files panel** at `/content/native_essex_export.zip`, or save it under the study's private `data/raw/essex_femoral_neck_loading/` folder, then rerun this notebook. Do not commit raw MAT files or the conversion ZIP to GitHub.

**If MATLAB is not available:** continue running Notebook 04 without the export ZIP. It will create a truthful pre-conversion report that records which scientific requirements remain blocked, rather than creating synthetic data.

In [ ]:
# ================== 04. Original MATLAB Export Availability ==================

# ------------ 1. Inspect the Version-Controlled Native Converter -------------
matlab_source = study_dir / "code" / "matlab" / "export_essex_native_tables.m"
if not matlab_source.is_file():
    raise FileNotFoundError("MATLAB converter missing: use a fresh GitHub clone.")
print("Native converter registered:", matlab_source.name)

# ---------- 2. Resolve a Single Optional Converted Numeric Archive -----------
native_archive = find_native_archive(raw_dir)
has_export = native_archive is not None
print("MATLAB-native ZIP supplied:", has_export)
if has_export:
    print("Source conversion archive SHA256:", sha256_file(native_archive))
else:
    print("Conversion status: AWAITING_NATIVE_EXPORT")
    print("No numeric source signals will be invented.")

**05. Safe Numeric Export Extraction and Record-Level Alignment**

A supplied export ZIP is checked against a strict manifest and approved folder structure, with limits on archive size, path traversal and duplicate members. The manifest must account for both `IMU` and `FE` for each of the 1,897 original source records. It is joined to the frozen approved registry using explicit original source keys, not tensor row order.

When a file is absent, the output tables intentionally report `awaiting_native_export`. There is no silent fallback to BoneNet's smaller derivative data.

In [ ]:
# ========== 05. Native Export Package, Checksums and Trial Identity ==========

# ---------- 1. Recover Approved Numeric Export Metadata If Supplied ----------
if has_export:
    validated_dir = unpack_native_archive(native_archive, export_dir)
    native_manifest = validate_native_manifest(
        validated_dir / "export_manifest.csv", registry
    )
    print("Native manifest source identities: VERIFIED")
else:
    validated_dir = None
    native_manifest = pd.DataFrame(
        columns=[*EXPORT_COLUMNS,"local_record_key"]
    )
native_manifest.to_csv(
    output_dir / "04_native_conversion_manifest.csv",index=False
)

# ---------------- 2. Describe the Observed Conversion Status -----------------
if has_export:
    conversion_status = (
        native_manifest.groupby(["modality","status"])
        .size().rename("records").reset_index()
    )
else:
    conversion_status = pd.DataFrame([
        {"modality":"IMU","status":"awaiting_native_export","records":1897},
        {"modality":"FE","status":"awaiting_native_export","records":1897},
    ])
conversion_status.to_csv(
    output_dir / "04_conversion_status_summary.csv",index=False
)
display(conversion_status)

**06. Finite Numerical Arrays and Native Column Labels**

A supplied MATLAB-v7 payload is loaded only after its source filename and manifest identity have been checked. The validator confirms finite numerical values, actual array shape, original table-column label count and SHA256 per file. Unsupported or missing FE data are **reported**, not imputed.

For scientific reporting, a candidate `101 × 42` IMU array, two-column `e1/e3` FE array or 11-position FE grid still requires original physical-unit definitions and independent temporal alignment. Numeric accessibility must not be labelled as confirmed biomechanical validity.

In [ ]:
# =========== 06. Native Numerical Payload and Integrity Validation ===========

# ------------- 1. Read All Real Numeric Payloads When Available --------------
payload_cols = [
    "local_record_key","subject_id","task","updated_trial_id",
    "modality","conversion_status","n_time","n_channels",
    "finite","label_count_matches","time_101_candidate",
    "fe_11_candidate","expected_imu_42_channels",
    "named_fe_e1_e3_candidate","sha256"
]
if has_export:
    payload_audit = validate_numeric_payloads(
        validated_dir,native_manifest
    )
    native_summary = conversion_summary(native_manifest,payload_audit)
    assert len(payload_audit)==len(native_manifest)
else:
    payload_audit = pd.DataFrame(columns=payload_cols)
    native_summary = {
        "manifest_rows":0, "exported_numeric_payloads":0,
        "validated_finite_payloads":0, "imu_time_101_candidate":0,
        "imu_42_channel_candidate":0, "fe_time_11_candidate":0,
        "fe_named_pair_candidate":0, "conversion_status_counts":{},
        "physical_units_verified":False,
        "imu_fe_temporal_alignment_verified":False,"model_ready":False
    }
payload_audit.to_csv(
    output_dir/"04_numeric_payload_validation.csv",index=False
)
display(pd.DataFrame([{k:v for k,v in native_summary.items()
                      if k!="conversion_status_counts"}]))

# ----------- 2. Enforce Separation of Numeric and Model Readiness ------------
assert not native_summary["model_ready"]
assert not native_summary["physical_units_verified"]
assert not native_summary["imu_fe_temporal_alignment_verified"]
print("Finite numerical arrays validated:",
      native_summary["validated_finite_payloads"])
print("Biomechanical and FE temporal semantics: NOT VERIFIED")

**07. Temporal Grid and FE Principal-Strain Semantics**

The original dataset describes inputs on a 101-position normalised movement grid; the FE analysis is derived from model calculations at a lower temporal density. We screen shapes and original source column labels, but **do not interpolate or invent FE values**.

The four indicators below have different meanings: 101-position numerical input, 42-channel IMU input, 11-position candidate FE output and original `e1/e3` two-column names. Even if all pass shape checks, units, exact FE calculation indices and corresponding input–target timing require a separate scientific validation.

In [ ]:
# ============== 07. Time Grid and Physical-Semantics Candidates ==============

# ------------ 1. Record Numeric Shape and Column-Name Candidates -------------
if has_export:
    finite_rows = payload_audit.loc[
        payload_audit["finite"] == True
    ].copy()
else:
    finite_rows = pd.DataFrame(columns=payload_cols)
finite_rows.to_csv(output_dir/"04_numeric_grid_candidates.csv",index=False)
time_audit = pd.DataFrame([
    {"criterion":"IMU sequence with 101 source positions",
     "candidates":native_summary["imu_time_101_candidate"],"verified":False},
    {"criterion":"IMU sequence with 42 source channels",
     "candidates":native_summary["imu_42_channel_candidate"],"verified":False},
    {"criterion":"FE sequence with 11 source positions",
     "candidates":native_summary["fe_time_11_candidate"],"verified":False},
    {"criterion":"FE channels explicitly named e1 and e3",
     "candidates":native_summary["fe_named_pair_candidate"],"verified":False},
])
display(time_audit)
time_audit.to_csv(output_dir/"04_temporal_semantics_audit.csv",index=False)

# ------------- 2. Preserve the Unresolved Scientific Boundaries --------------
print("IMU channel units and anatomical axes: UNVERIFIED")
print("FE e1/e3 sign and physical units: UNVERIFIED")
print("Native FE sampling and paired timebase: UNVERIFIED")
print("No FE interpolation, scalar imputation or model training performed.")

**08. Descriptive Research Figure and Authentic Numeric Example**

The first panel reports original-source FE field availability across six activities. The second panel reports actual conversion statuses, not prediction results. A third panel plots the **real first column** of one successfully validated numerical IMU export only when such a file exists; otherwise it states that the conversion archive is absent. Any displayed numerical trajectory is labelled as physically **unverified**, with no manufactured waveform.

In [ ]:
# =========== 08. Original Source Coverage and Native Export Figure ===========

# ----------- 1. Plot Published FE Representation and Export Status -----------
fig,axes=plt.subplots(3,1,figsize=(10,11),constrained_layout=True)
original_fe.pivot(
    index="task",columns="fe_representation",values="records"
).fillna(0).astype(int).sort_index().plot(
    kind="bar",stacked=True,ax=axes[0],rot=25
)
axes[0].set_title("Original Essex FE field representation by activity")
axes[0].set_ylabel("Original movement records")
axes[0].set_xlabel("Source activity")

status_totals=conversion_status.groupby("status")["records"].sum()
axes[1].bar(status_totals.index,status_totals.values)
axes[1].set_title("Native numeric conversion status")
axes[1].set_ylabel("Record-modality entries")
axes[1].tick_params(axis="x",labelrotation=20)

# ------------ 2. Plot Only Real Source Numeric Data When Present -------------
imu_exports = (
    native_manifest.loc[
        (native_manifest["modality"]=="IMU")
        &(native_manifest["status"]=="exported")
    ] if has_export else pd.DataFrame()
)
if len(imu_exports):
    row=imu_exports.iloc[0]
    numeric=loadmat(
        validated_dir/row["export_relpath"],
        variable_names=["values"]
    )["values"]
    axes[2].plot(np.arange(numeric.shape[0]),numeric[:,0])
    axes[2].set_title("First original numeric IMU channel (units unverified)")
    axes[2].set_xlabel("Source sample index")
    axes[2].set_ylabel("Numerical value")
else:
    axes[2].text(0.5,0.5,"Original MATLAB-native IMU export unavailable",
                 ha="center",va="center",transform=axes[2].transAxes)
    axes[2].set_title("Authentic numerical waveform availability")
    axes[2].set_xticks([])
    axes[2].set_yticks([])
save_figure(fig,output_dir/"04_native_conversion_evidence.png")
plt.show()

**09. Scientific Quality Gate and Provenance Record**

The data-source gate retains its previous PASS. Numeric extraction is graded independently as `AWAITING_NATIVE_EXPORT` or `NUMERIC_EXPORT_VALIDATED`. **Neither state is equivalent to model readiness.** An exported numerical matrix can still lack verified channel identities, physical units, source FE sign convention and correct IMU/FE temporal pairing.

The actual MATLAB conversion archive SHA256 (when supplied) is recorded separately from the Figshare original archive fingerprint to preserve provenance.

In [ ]:
# ============== 09. Original Source and Conversion Quality Gate ==============

# ----------- 1. Enforce Source Identity and Scientific Separation ------------
assert len(registry)==1897 and registry["subject_id"].nunique()==40
assert prior02["archive_sha256"]==prior03["archive_sha256"]
assert int(original_fe["records"].sum())==len(registry)
assert not native_summary["model_ready"]
conversion_state=(
    "NUMERIC_EXPORT_VALIDATED" if has_export else "AWAITING_NATIVE_EXPORT"
)
gate = {
    "source_article":30225739,
    "source_version":13,
    "original_matlab_zip_sha256":prior03["archive_sha256"],
    "source_record_count":1897,"participant_count":40,
    "fe_empty_source_records":168,
    "native_export_supplied":bool(has_export),
    "native_export_sha256":sha256_file(native_archive) if has_export else None,
    "original_source_gate":"PASSED",
    "native_conversion_gate":conversion_state,
    "numeric_export_audit":native_summary,
    "physical_imu_units_verified":False,
    "physical_fe_units_verified":False,
    "fe_timestep_alignment_verified":False,
    "model_ready":False,
    "model_training_performed":False,
}
(output_dir/"04_native_conversion_quality_gate.json").write_text(
    json.dumps(gate,indent=2)+"\n",encoding="utf-8"
)
print("ORIGINAL SOURCE INTEGRITY: PASSED")
print("NATIVE NUMERIC CONVERSION:",conversion_state)
print("SUPERVISED REGRESSION READINESS: NOT VERIFIED")

**10. Scientific Evidence Export and Next Boundary**

Store eight `04_*` research outputs: the source conversion contract, source-linked conversion manifest, conversion statuses, numeric payload validity, grid candidates, semantic checks, one descriptive PNG and the JSON Quality Gate report.

**Interpretation:** A Colab run can legitimately complete with `AWAITING_NATIVE_EXPORT` and `model_ready=false`. This records the outstanding independent MATLAB-native conversion requirement rather than implying successful numerical recovery. Exported original numeric MAT payloads and the transfer ZIP must remain under private raw-data storage, not GitHub. After execution, the final SVG is added with a title **without stage numbering** and the notebook commentary is updated to the actual evidence.

In [ ]:
# ============= 10. Research Output Registry and Colab Packaging ==============

# --------------------- 1. Check Required Evidence Files ----------------------
expected=[
    "04_expected_conversion_contract.csv",
    "04_native_conversion_manifest.csv",
    "04_conversion_status_summary.csv",
    "04_numeric_payload_validation.csv",
    "04_numeric_grid_candidates.csv",
    "04_temporal_semantics_audit.csv",
    "04_native_conversion_evidence.png",
    "04_native_conversion_quality_gate.json",
]
missing=[name for name in expected if not (output_dir/name).is_file()]
if missing:
    raise FileNotFoundError("Missing conversion audit files: "+", ".join(missing))
print("Notebook 04 evidence files:",len(expected))

# -------- 2. Package Only Summary Outputs, Never Source Numeric Data ---------
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output",stage="04"
)